# Benchmarking of lava flow models -- Realistic test case -- Mauna Loa 2022 eruption
This notebook compares several forward models of lava flow emplacement using the 2022 eruption of Mauna Loa as a test case. The input to all the models include the topography of Mauna Loa, the location of the main vent (fissure 3), and other parameters required for each model, such as volume, duration, or effusion rate. All of the other required input parameters have been pre-stored in the input files for each model.  

# First, import the required libraries

In [2]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import subprocess
import os
import utm
import xarray as xr
import rasterio as rio
from pyproj import Transformer
import sys
import glob
import geopandas as gpd
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor

<h3>Common Parameters</h3>
Here we provide the topography file, eruption volume, and vent location. 
This workflow expects a lat/long or UTM referenced GeoTIFF.

In [3]:
easting = 234392

northing = 2159562

# Total volume in m^3 from eruption
volume = 2e8

# Name to reference files
name = "MaunaLoa"

# IMEX and Lava2d are depth-averaged time-dependent models, and thus require an approximate run time.
# Below, we assign a reasonable yet arbitrary time (3 hour) for the flow to run (in seconds)
# Please change it at your leisure
run_time = 3600

# Please enter 1, 2, or 4 (original resolution, half, quarter respectively)
downscale = 2

# Full path to your DEM file (.tif/.tiff or .asc/.ascii)
dem = "/home/jovyan/Lava Benchmark/DEMs_Lava_Benchmark/MaunaLoa.tif"

gdf = gpd.read_file("flowoutline.shp")

DataSourceError: flowoutline.shp: No such file or directory

<h4> The following cell re-formats the DEMs to ensure we provide each model with the appropriate format </h4>

In [40]:
raster = rxr.open_rasterio(dem)
raster = raster.sel({"band": 1})
output_num = str(int(run_time/100))
out = output_num.zfill(4)

filename, file_extension = os.path.splitext(dem)
file_name = os.path.basename(dem)
file_name = os.path.splitext(file_name)[0]
type_flag = 0 #0 is ascii, 1 is tiff
if file_extension in [".geotiff", ".tiff", ".tif"]:
    type_flag = 1
    asc_name = "".join((name,".asc"))
    tiff_name = "".join((name, ".tiff"))
    victor.tiff_to_utm_ascii(dem,asc_name)
    # victor.convert_dem_format(dem,tiff_name, output_format="geotiff")
elif file_extension in [".asc", ".ascii"]:
    type_flag = 0
    tiff_name = "".join((file_name, ".tiff"))
    asc_name = "".join((file_name,".asc"))
    raster.rio.to_raster(tiff_name)


<h2> Before starting the computations, confirm the location of the vent we want to use. </h2>

In [6]:
os.chdir('/home/jovyan/Lava Benchmark/')
rstr = rio.open(asc_name)
bounds = rstr.bounds

xll, yll = int(bounds.left), int(bounds.bottom)
r = rio.open(dem)
array = rxr.open_rasterio(asc_name)
res = array.rio.resolution()
vent_easting, vent_northing = easting, northing
coordinates = np.array([easting, northing])
victor.plot_dem(asc_name, coordinates)

In [42]:
os.chdir('/home/jovyan/Lava Benchmark/')
rstr = rio.open(asc_name)
rstr_tiff = rio.open(tiff_name)
bounds = rstr.bounds
crs = rstr_tiff.crs
res = rstr.res
transformer = Transformer.from_crs(crs.to_string(), "EPSG:4326")
transformer2 = Transformer.from_crs("EPSG:4326", crs.to_string())


### Now, we can run everything. Press run on every cell until the next text box.

Keep in mind, the time dependent models can take a long time, so sit back and relax, though keep an eye out for your results. Each model will also display its individual result before the final comparison.

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/Molasses")
f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = "MIN_RESIDUAL = .55\n"
conf[3] = "MAX_RESIDUAL = 9\n"
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/1000}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/1000}\n"""
conf[10] = f"""DEM_FILE = {asc_name}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()
if not os.path.exists(f"./{asc_name}"):
    subprocess.run(f"""ln -s ../{asc_name} .""",shell=True)
subprocess.run("./molasses_2022 custom_molasses.conf",shell=True, stderr=subprocess.DEVNULL)

victor.convert_molasses(f"molasses_{name}",resolution=30)

In [ ]:
victor.plot_flow("./MaunaLoa.asc",f"./molasses_{name}.asc",zoom=False)

### MrLavaLoba: Additional Notes

MrLavaLoba input parameters are stored in `input_data.py` in the `mr_lava_loba` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/mr_lava_loba/')
subprocess.run("mv mr_lava_loba_maunaloa.inp mr_lava_loba.inp", shell=True)
g=open("mr_lava_loba.inp","r+")
inp = g.readlines()
inp[1] = f""" RUN_NAME="{name}",\n"""
inp[2] = f""" SOURCE="../DEMs_Lava_Benchmark/MaunaLoa.asc"\n"""
inp[13] = f" X_VENT= {easting},\n"
inp[14] = f" Y_VENT= {northing},\n"
inp[26] = f" TOTAL_VOLUME={volume},\n"

g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("mr_lava_loba",shell=True)
subprocess.run("mv mr_lava_loba.inp mr_lava_loba_maunaloa.inp", shell=True)

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/mr_lava_loba/')
mll_outputs = sorted(glob.glob(f'{name}*thickness_full.asc'))
mll_out = mll_outputs[-1]
victor.plot_flow(asc_name,mll_out,zoom=False)

### IMEX_Lava: Additional Notes

IMEX_Lava input parameters are stored in `IMEX_LavaFlow.inp` in the `IMEX` folder. This is a physics-based, highly accurate, slower model. If desiring to change additional input parameters, we recommend editing the `TEMPERATURE_PARAMETERS`, `RHEOLOGY_PARAMETERS`, `GAS_TRANSPORT_PARAMETERS`, and `LIQUID_TRANSPPORT_PARAMETERS` namelists as needed. All other sections are carefully prepared based in your initial basic inputs, so ignore all other sections.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/IMEX_LavaFlow")
subprocess.run("mv IMEX_LavaFlow_maunaloa.inp IMEX_LavaFlow.inp", shell=True)

h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[1] = f''' RUN_NAME="{name}"\n'''
inp[4] = f""" T_END=  {run_time},\n"""
inp[5] = f""" DT_OUTPUT=  1000,\n"""
inp[13] = f""" TOPOGRAPHY_FILE="MaunaLoa.asc"                  ,\n"""
inp[14] = f""" X0= {int(xll)+res[0]}     ,\n"""
inp[15] = f""" Y0= {int(yll)+res[0]}     ,\n"""
inp[16] = f""" COMP_CELLS_X=  {int(len(array.x)/downscale)-5},\n"""
inp[17] = f""" COMP_CELLS_Y=  {int(len(array.y)/downscale)-5},\n"""
inp[18] = f""" CELL_SIZE = {res[0]*downscale}D0 ,\n"""
inp[48] = f"""  X_SOURCE=  {int(vent_easting)}.0D0 ,\n"""
inp[49] = f"""  Y_SOURCE=  {int(vent_northing)}.0D0 ,\n"""
inp[50] = f"""  R_SOURCE=  {res[0]}D0     ,\n"""
inp[51] = f""" VEL_SOURCE= {float(volume)/(res[0]**2*np.pi*run_time)} \n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')
subprocess.run("mv IMEX_LavaFlow.inp IMEX_LavaFlow_maunaloa.inp", shell=True)

os.chdir("/home/jovyan/Lava Benchmark")

In [ ]:
victor.plot_flow("MaunaLoa.asc", f"{name}_max.asc",zoom=False)

### LAVA2D: Additional Notes

LAVA2D input parameters are stored in `input.py` in the `lava2d` folder. This is a physics-based, highly accurate, slower model. If desiring to change additional input parameters, we recommend editing the `set_vent_props`, `set_lava_props`, `set_rheo`, `set_ambient`, and `set_numeric` sections as needed. All other sections are carefully prepared based in your initial basic inputs, so please leave them as is.

In [1]:
os.chdir("/home/jovyan/Lava Benchmark/lava2d")
left, bottom, right, top = r.bounds
lower_left = transformer.transform(left, bottom)
top_right = transformer.transform(right, top)
src = transformer.transform(vent_easting, vent_northing)

l = open("input.py","r+")
inp = l.readlines()

if type_flag == 0:
    inp[4] = f"""    path_to_dem_file    = ('/home/jovyan/shared-readwrite/DEMs/MaunaLoa.tif'),\n"""
else:
    inp[4] = f"""    path_to_dem_file    = ('/home/jovyan/shared-readwrite/DEMs/MaunaLoa.tif'),\n"""
inp[5] = f"""    Lon_SRC             = {src[1]}, # source longitude\n"""
inp[6] = f"""    Lat_SRC             = {src[0]},    # source latitude\n"""
inp[7] = f"""    Lon_LowerLeft       = {left}, # bounding box: lower-left longitude\n"""
inp[8] = f"""    Lat_LowerLeft       = {bottom}, # bounding box: lower-left latitude\n"""
inp[9] = f"""    Lon_UpperRight      = {right}, # bounding box: upper-right longitude\n"""
inp[10] = f"""    Lat_UpperRight      = {top},   # bounding box: upper-right latitude\n"""
inp[12] = f"""    dx_desired          = {res[0]*downscale},          # meters\n"""
inp[74] = f"""    max_iter = None, #one of them can be none, so the default value for both is none, if both none run till killed\n"""
inp[75] = f"""    max_time_hr = {run_time/3600},\n"""
l.seek(0)
l.writelines(inp)
l.truncate()
l.close()

k=open("./example_vents/vent_01.txt","r+")
inp = k.readlines()
inp[1] = f'''0	0	0	100	-75	20	{volume/(run_time*50)}\n'''
inp[2] = f'''10000	0	0	100	-75	20	{volume/(run_time*50)}\n'''

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

subprocess.run("python input.py",shell=True)

NameError: name 'os' is not defined

In [ ]:
src = transformer.transform(vent_easting, vent_northing)
r = rio.open("hawaii.tif")
left, bottom, right, top = r.bounds
r.bounds

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/lava2d")

subprocess.run("python convert_output.py outputs/MaunaLoa/out.nc -o lava2d_maunaloa.asc", shell=True)

In [ ]:
victor.plot_flow("../MaunaLoa.asc", f"{name}_output.tiff")

### Finally, we can take a look at all the models side by side.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

r = rxr.open_rasterio(f"IMEX_LavaFlow/{name}_max.asc",masked=True)
r2 = rxr.open_rasterio(f"Molasses/molasses_{name}.asc",masked=True)
r3 = rxr.open_rasterio(f"mr_lava_loba/{mll_out}",masked=True)
r4 = rxr.open_rasterio(f"lava2d/{name}_output.tiff",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min(), r4.max()

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (9,9))
gdf.plot(ax=ax0)
gdf.plot(ax=ax1)
gdf.plot(ax=ax2)
gdf.plot(ax=ax3)
thickness0 = victor.plot_flow(asc_name, f"Molasses/molasses_{name}.asc", axes=ax0,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(asc_name, f"mr_lava_loba/{mll_out}", axes=ax1, title='MrLavaLoba',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(asc_name, f"IMEX_LavaFlow/{name}_max.asc", axes=ax2,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow(asc_name, f"lava2d/{name}_output.tiff", axes=ax3 ,title='Lava2D', minimum=1e-6, scale=maximum)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")

for ax in [ax0, ax1, ax2, ax3]:
    ax.set_aspect('equal')
    ax.set_box_aspect(1)

In [ ]:
import helpers

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 8))

os.chdir("/home/jovyan/Lava Benchmark")
victor.standardize_raster('./lava2d/MaunaLoa_output.tiff', './main_flow.tif')

raster = rxr.open_rasterio("./union_MaunaLoa_output.tiff",masked=True)
raster_vals = (raster > 0).values
raster2 = rxr.open_rasterio("./union_main_flow.tif",masked=True)
raster2_vals= (raster2 > 0).values

out = helpers.best_fit_s2m(raster_vals, raster2_vals)
raster2.values = raster2_vals
raster2.plot(cmap="gray",add_colorbar=False,ax=ax1)
raster.plot(ax=ax1)
ax1.set_title("Lava2d")

victor.standardize_raster('./molasses_MaunaLoa.tiff', './main_flow.tif')

raster = rxr.open_rasterio("./union_molasses_MaunaLoa.tiff",masked=True)
raster = raster.rio.write_crs("EPSG:32605")
raster_vals = (raster > 0).values
raster2 = rxr.open_rasterio("./union_main_flow.tif",masked=True)
raster2_vals= (raster2 > 0).values

out = helpers.best_fit_s2m(raster_vals, raster2_vals)
raster2.values = raster2_vals
raster2.plot(cmap="gray",add_colorbar=False,ax=ax2)
raster.plot(ax=ax2)
ax2.set_title("MOLASSES")

victor.standardize_raster('./mll_maunaloa.tiff', './main_flow.tif')

raster = rxr.open_rasterio("./union_mll_maunaloa.tiff",masked=True)
raster = raster.rio.write_crs("EPSG:32605")
raster_vals = (raster > 0).values
raster2 = rxr.open_rasterio("./union_main_flow.tif",masked=True)
raster2_vals= (raster2 > 0).values

out = helpers.best_fit_s2m(raster_vals, raster2_vals)
raster2.values = raster2_vals
raster2.plot(cmap="gray",add_colorbar=False,ax=ax3)
raster.plot(ax=ax3)
ax3.set_title("MrLavaLoba")


victor.standardize_raster('./imex_lavaflow_MaunaLoa.tiff', './main_flow.tif')

raster = rxr.open_rasterio("./union_imex_lavaflow_MaunaLoa.tiff",masked=True)
raster = raster.rio.write_crs("EPSG:32605")
raster_vals = (raster > 0).values
raster2 = rxr.open_rasterio("./union_main_flow.tif",masked=True)
raster2_vals= (raster2 > 0).values

out = helpers.best_fit_s2m(raster_vals, raster2_vals)
raster2.values = raster2_vals
raster2.plot(cmap="gray",add_colorbar=False,ax=ax4)
raster.plot(ax=ax4)
ax4.set_title("IMEX_LavaFlow")